In [0]:
%sql
create catalog if not exists banking;

In [0]:
%sql
create schema if not exists banking.metadata;
create schema if not exists banking.silver;
create schema if not exists banking.gold;

In [0]:
%sql
create table if not exists banking.metadata.tables(
table_id int, 
table_name string, 
source_system string, 
source_schema string, 
source_table string, 
source_path string, 
target_layer string, 
bronze_schema string, 
silver_schema string, 
gold_schema string, 
active_flag boolean,  
load_order string, 
created_at timestamp 
) using delta;

In [0]:
%sql
create table if not exists banking.metadata.table_watermarks(
  table_id int, 
  last_watermark_value string, 
  last_updated_at timestamp,
  last_run_id bigint
)using delta partitioned by(table_id);

In [0]:
%sql
create table if not exists banking.metadata.pipeline_runs(
  run_id bigint, 
  table_id int, 
  layer string,
  start_time timestamp,
  end_time timestamp,
  status string,
  number_of_records bigint,
  error_message string
)using delta partitioned by(table_id);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS banking.metadata.table_parameters (
    table_id            INT,
    parameter_name      STRING,        -- load_type / primary_key / watermark_column
    parameter_value     STRING,
    created_at          TIMESTAMP
)
USING DELTA;

In [0]:
%sql
--  Create Volume
create schema if not exists banking.source;

create volume if not exists banking.source.volume;



In [0]:
%sql

INSERT INTO banking.metadata.table_parameters VALUES




-- ================= CUSTOMERS =================
(1, 'load_type', 'MERGE', current_timestamp()),
(1, 'primary_key', 'customer_id', current_timestamp()),
(1, 'watermark_column', 'updated_at', current_timestamp()),

-- ================= ACCOUNTS =================
(2, 'load_type', 'MERGE', current_timestamp()),
(2, 'primary_key', 'account_id', current_timestamp()),
(2, 'watermark_column', 'updated_at', current_timestamp()),

-- ================= TRANSACTIONS =================
(3, 'load_type', 'APPEND', current_timestamp()),
(3, 'primary_key', 'txn_id', current_timestamp()),
(3, 'watermark_column', 'txn_timestamp', current_timestamp()),

-- ================= BRANCHES =================
(4, 'load_type', 'FULL', current_timestamp()),
(4, 'primary_key', 'branch_code', current_timestamp()),

-- ================= CREDIT BUREAU REPORTS =================
(5, 'load_type', 'MERGE', current_timestamp()),
(5, 'primary_key', 'customer_id', current_timestamp()),
(5, 'watermark_column', 'bureau_pull_date', current_timestamp()),

-- ================= PAYMENT GATEWAY LOGS =================
(6, 'load_type', 'APPEND', current_timestamp()),
(6, 'primary_key', 'txn_id', current_timestamp()),
(6, 'watermark_column', 'processed_timestamp', current_timestamp());

In [0]:
%sql
truncate table banking.metadata.tables

In [0]:
%sql
--table_id int, 
-- table_name string, 
-- source_system string, 
-- source_schema string, 
-- source_table string, 
-- source_path string, 
-- target_layer string, 
-- bronze_schema string, 
-- silver_schema string, 
-- gold_schema string, 
-- active_flag boolean,  
-- load_order string, 
-- created_at timestamp

INSERT INTO banking.metadata.tables VALUES
-- 1. Customers (SQL Server)
(1, 'customers', 'table', 'banking', 'customers', 'banking.source.customers', 'silver', 'bronze', 'silver', NULL, TRUE, 1, current_timestamp()),

-- 2. Accounts (SQL Server)
(2, 'accounts', 'table', 'banking', 'accounts', 'banking.source.accounts', 'silver', 'bronze', 'silver', NULL, TRUE, 2, current_timestamp()),

-- 3. Transactions (SQL Server)
(3, 'transactions', 'table', 'banking', 'transactions', 'banking.source.transactions', 'silver', 'bronze', 'silver', NULL, TRUE, 3, current_timestamp()),

-- 4. Branches (SQL Server - Full Load)
(4, 'branches', 'table', 'banking', 'branches', 'banking.source.branches', 'silver', 'bronze', 'silver', NULL, TRUE, 4, current_timestamp()),

-- 5. Credit Bureau Reports (Blob CSV)
(5, 'credit_bureau_reports', 'blob', NULL, NULL,
 '/Volumes/banking/source/volume/credit_bureau_reports/', 'silver',
 'bronze', 'silver', NULL, TRUE, 5, current_timestamp()),

-- 6. Payment Gateway Logs (Blob CSV)
(6, 'payment_gateway_logs', 'blob', NULL, NULL,
 '/Volumes/banking/source/volume/payment_gateway_logs/', 'silver',
 'bronze', 'silver', NULL, TRUE, 6, current_timestamp());

In [0]:
%sql
INSERT INTO banking.metadata.table_watermarks VALUES
(1, '1900-01-01 00:00:00', current_timestamp(), NULL),
(2, '1900-01-01 00:00:00', current_timestamp(), NULL),
(3, '1900-01-01 00:00:00', current_timestamp(), NULL),
(5, '1900-01-01 00:00:00', current_timestamp(), NULL),
(6, '1900-01-01 00:00:00', current_timestamp(), NULL);

In [0]:
%sql
INSERT INTO banking.metadata.tables
VALUES (
    7,
    'customer_360',
    'silver',
    NULL,
    NULL,
    NULL,
    'gold',
    NULL,
    NULL,
    'gold',
    TRUE,
    1,
    current_timestamp()
);

-- COMMAND ----------

INSERT INTO banking.metadata.tables
VALUES
(
8,
'branch_performance',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
2,
current_timestamp()
),

(
9,
'transaction_channel_summary',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
3,
current_timestamp()
),

(
10,
'daily_bank_kpi',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
4,
current_timestamp()
);

-- COMMAND ----------

INSERT INTO banking.metadata.tables
VALUES (
    11,
    'risk_customer_summary',
    'silver',
    NULL,
    NULL,
    NULL,
    'gold',
    NULL,
    NULL,
    'gold',
    TRUE,
    1,
    current_timestamp()
);